# Código para tese 

In [1]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from scipy.stats import entropy
import networkx as nx
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
#!pip install openpyxl

In [3]:
projects_df = pd.read_excel("/home/20240117/tese/data/project.xlsx")
organization_df = pd.read_excel("/home/20240117/tese/data/organization.xlsx")
topics_df = pd.read_excel("/home/20240117/tese/data/topics.xlsx")
organization_df = organization_df.dropna(how="all").copy()
organization_df = organization_df[organization_df["projectID"].notna()].copy()

In [4]:
merged_df = organization_df.merge(projects_df, left_on="projectID", right_on="id", how="left")

In [5]:
print(merged_df.columns)

Index(['projectID', 'projectAcronym', 'organisationID', 'vatNumber', 'name',
       'shortName', 'SME', 'activityType', 'street', 'postCode', 'city',
       'country', 'nutsCode', 'geolocation', 'organizationURL', 'contactForm',
       'contentUpdateDate_x', 'rcn_x', 'order', 'role', 'ecContribution',
       'netEcContribution', 'totalCost_x', 'endOfParticipation', 'active',
       'id', 'acronym', 'status', 'title', 'startDate', 'endDate',
       'totalCost_y', 'ecMaxContribution', 'legalBasis', 'topics',
       'ecSignatureDate', 'frameworkProgramme', 'masterCall', 'subCall',
       'fundingScheme', 'objective', 'contentUpdateDate_y', 'rcn_y',
       'grantDoi', 'keywords'],
      dtype='str')


In [6]:
cols_to_keep = [
    'projectID', 'projectAcronym', 'organisationID', 'name', 'SME', 'activityType',
    'country', 'nutsCode', 'geolocation', 'ecContribution', 'totalCost_x', 'role',
    'title', 'startDate', 'endDate', 'totalCost_y', 'ecMaxContribution',
    'objective', 'keywords', 'frameworkProgramme', 'topics', 'fundingScheme',
    'masterCall'
]

clean_df = merged_df[cols_to_keep].copy()

In [7]:
clean_df 


,projectID,projectAcronym,organisationID,name,SME,activityType,country,nutsCode,geolocation,ecContribution,...,startDate,endDate,totalCost_y,ecMaxContribution,objective,keywords,frameworkProgramme,topics,fundingScheme,masterCall
0,101201611,COOLIO,999869114,UNIVERSITAET INNSBRUCK,False,HES,AT,AT332,"47.2629172,11.3843965",2500000.00,...,2025-09-01,2030-08-31,0,2500000,Quantum physics lies at the heart of many phen...,"quantum simulation, quantum dynamics, quantum...",HORIZON,ERC-2024-ADG,HORIZON-ERC,ERC-2024-ADG
1,101198761,MINTRAF,998922879,CENTRO NACIONAL DE INVESTIGACIONES CARDIOVASCU...,False,REC,ES,ES300,"40.47668785,-3.696091431284352",2455167.00,...,2025-09-01,2030-08-31,0,2455167,Mitochondrial research has consistently yielde...,"Mitochondria, OxPhos, Cardiovascular, Heter...",HORIZON,ERC-2024-ADG,HORIZON-ERC,ERC-2024-ADG
2,101198411,StereoCPC,999907720,TECHNION - ISRAEL INSTITUTE OF TECHNOLOGY,False,HES,IL,IL,"32.8191218,34.9983856",2500000.00,...,2025-09-01,2030-08-31,0,2500000,Stereoselective synthesis is a central aspect ...,"Non-classical carbocation, stereochemistry, ac...",HORIZON,ERC-2024-ADG,HORIZON-ERC,ERC-2024-ADG
3,101199868,DM-Dawn,999596544,PHYSIKALISCH-TECHNISCHE BUNDESANSTALT,False,REC,DE,DE911,"52.2950801,10.4578976486826",399668.75,...,2025-09-01,2030-08-31,0,2271204,What is the nature of dark matter (DM)? This ...,"Nuclear clock, ultralight DM",HORIZON,ERC-2024-ADG,HORIZON-ERC,ERC-2024-ADG
4,101199868,DM-Dawn,999979306,WEIZMANN INSTITUTE OF SCIENCE,False,HES,IL,IL,"31.8952532,34.8105616",1529800.25,...,2025-09-01,2030-08-31,0,2271204,What is the nature of dark matter (DM)? This ...,"Nuclear clock, ultralight DM",HORIZON,ERC-2024-ADG,HORIZON-ERC,ERC-2024-ADG
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
116462,101132079,Kaleidos,915420526,WEDO PROJECT INTELLIGENCE MADE EASYSL,True,PRC,ES,ES511,"41.370449300000004,2.1500219563810696",298625.00,...,2024-02-01,2026-07-31,1069225,1069225,KALEIDOS brings together research and innovati...,"Knowledge Valorisation, Open Science, Quadrupl...",HORIZON,HORIZON-WIDERA-2023-ERA-01-03,HORIZON-CSA,HORIZON-WIDERA-2023-ERA-01
116463,101132079,Kaleidos,988004074,FUNDACIO PRIVADA PARC DE RECERCA UAB,False,OTH,ES,ES511,"41.4910324,2.1374969",0.00,...,2024-02-01,2026-07-31,1069225,1069225,KALEIDOS brings together research and innovati...,"Knowledge Valorisation, Open Science, Quadrupl...",HORIZON,HORIZON-WIDERA-2023-ERA-01-03,HORIZON-CSA,HORIZON-WIDERA-2023-ERA-01
116464,101132079,Kaleidos,999993953,ALMA MATER STUDIORUM - UNIVERSITA DI BOLOGNA,False,HES,IT,ITH55,"44.4968718,11.3524529",144673.75,...,2024-02-01,2026-07-31,1069225,1069225,KALEIDOS brings together research and innovati...,"Knowledge Valorisation, Open Science, Quadrupl...",HORIZON,HORIZON-WIDERA-2023-ERA-01-03,HORIZON-CSA,HORIZON-WIDERA-2023-ERA-01
116465,101132079,Kaleidos,999986484,UNIVERSITAT AUTONOMA DE BARCELONA,False,HES,ES,ES511,"41.4910324,2.1374969",183762.50,...,2024-02-01,2026-07-31,1069225,1069225,KALEIDOS brings together research and innovati...,"Knowledge Valorisation, Open Science, Quadrupl...",HORIZON,HORIZON-WIDERA-2023-ERA-01-03,HORIZON-CSA,HORIZON-WIDERA-2023-ERA-01


In [8]:
topics_to_devide_df = clean_df.merge(
    topics_df[['projectID', 'topic', 'title']].rename(columns={'title': 'topic_title'}),
    on='projectID',
    how='left'
)

topic_map = {
    "Health": ["health", "medical", "medicine", "hospital", "public health"],
    "Agriculture": ["agriculture", "rural", "farming", "food"],
    "Energy": ["energy", "renewable", "power", "electricity", "fuel"],
    "Environment": ["environment", "climate", "biodiversity", "nature"],
    "Transport": ["transport", "mobility", "rail", "road", "aviation"],
    "Digital": ["digital", "AI", "ICT", "big data", "cyber"],
    "Security": ["security", "border", "defence", "safety"],
    "Culture": ["culture", "media", "heritage", "creative"],
    "Education": ["education", "training", "skills"],
    "Research": ["research", "innovation", "science"],
    "Space": ["space", "satellite", "astronomy"]
}

def classify_topic(text):
    if pd.isna(text):
        return "Unknown"
    txt = text.lower()
    for category, keywords in topic_map.items():
        if any(word in txt for word in keywords):
            return category
    return "Other"

topics_to_devide_df["broad_topic"] = topics_to_devide_df["topic_title"].apply(classify_topic)

#topics mapping

mapping = {
"ERC ADVANCED GRANTS": "Ciências Fundamentais",
"ERC PROOF OF CONCEPT GRANTS": "Ciências Fundamentais",
"ERC STARTING GRANTS": "Ciências Fundamentais",
"ERC CONSOLIDATOR GRANTS": "Ciências Fundamentais",
"MSCA Staff Exchanges 2024": "Educação & Formação",
"MSCA Postdoctoral Fellowships 2023": "Educação & Formação",
"MSCA Postdoctoral Fellowships 2024": "Educação & Formação",
"EIC Pathfinder Open 2021": "Inovação & Tecnologias Emergentes",
"Tackling European skills and labour shortages": "Educação & Trabalho",
"Market Uptake Measures of renewable energy systems": "Energia",
"Advanced imaging and sensing technologies (IA)(Photonics Partnership)": "Digital & Indústria",
"Biodiversity, economics and finance: unlocking financial flows towards reversing of biodiversity loss": "Clima & Ambiente",
"Autonomous systems used for infrastructure protection": "Segurança & Defesa",
"Design for adaptability, re-use and deconstruction of buildings, in line with the principles of circular economy (Built4People Partnership)": "Construção Sustentável",
"Strengthening the sustainability and resilience of EU space infrastructure": "Espaço",
"Development of next generation synthetic renewable fuel technologies": "Energia",
"Innovative digital health solutions for sub-Saharan Africa": "Saúde",
"Support to the activities of the SET Plan Key Action area Renewable fuels and bioenergy": "Energia",
}

def classify_keywords(text):
    if not isinstance(text, str):
        return None
    text = text.lower()
    if any(word in text for word in ["health", "hlth", "hospital", "disease"]):
        return "Saúde"
    if any(word in text for word in ["energy", "renewable", "biofuel", "fuel"]):
        return "Energia"
    if any(word in text for word in ["education", "training", "skills", "school"]):
        return "Educação & Formação"
    if any(word in text for word in ["digital", "ai", "photonics", "industry", "ict"]):
        return "Digital & Indústria"
    if any(word in text for word in ["climate", "biodiversity", "environment", "sustainability"]):
        return "Clima & Ambiente"
    if any(word in text for word in ["transport", "mobility"]):
        return "Mobilidade"
    if any(word in text for word in ["security", "defence", "autonomous"]):
        return "Segurança & Defesa"
    if "space" in text:
        return "Espaço"
    return None

categories = {
"Saúde": "health medicine disease hospital clinical digital health",
"Energia": "renewable energy biofuel fuel solar wind hydrogen",
"Educação & Formação": "education training skills school university mobility research",
"Digital & Indústria": "digital ai industry photonics ict robotics automation",
"Clima & Ambiente": "climate environment biodiversity sustainability green",
"Mobilidade": "mobility transport vehicle logistics",
"Segurança & Defesa": "security defence autonomous protection infrastructure",
"Espaço": "space satellite astronomy exploration",
"Construção Sustentável": "construction building circular economy adaptability reuse",
"Inovação & Tecnologias Emergentes": "innovation emerging technologies breakthrough science",
"Ciências Fundamentais": "fundamental research basic science discovery physics chemistry biology",
}

In [9]:
vectorizer = TfidfVectorizer()
category_names = list(categories.keys())
category_docs = list(categories.values())
category_vectors = vectorizer.fit_transform(category_docs)

def classify_nlp(text):
    if not isinstance(text, str) or not text.strip():
        return "Outros"
    text_vector = vectorizer.transform([text])
    sims = cosine_similarity(text_vector, category_vectors)
    best_idx = int(sims.argmax())
    return category_names[best_idx]

def build_text(row):
    parts = []
    for col in ["topic_title", "keywords", "objective"]:
        v = row.get(col, None)
        if isinstance(v, str) and v.strip():
            parts.append(v)
    return " ".join(parts)

def classify_row(row):
    title = row.get("topic_title", None)

    if isinstance(title, str) and title in mapping:
        return mapping[title]

    text_combined = build_text(row)
    area = classify_keywords(text_combined)
    if area:
        return area

    return classify_nlp(text_combined)

topics_to_devide_df["Área"] = topics_to_devide_df.apply(classify_row, axis=1)

In [10]:
topics_to_devide_df.columns

Index(['projectID', 'projectAcronym', 'organisationID', 'name', 'SME',
       'activityType', 'country', 'nutsCode', 'geolocation', 'ecContribution',
       'totalCost_x', 'role', 'title', 'startDate', 'endDate', 'totalCost_y',
       'ecMaxContribution', 'objective', 'keywords', 'frameworkProgramme',
       'topics', 'fundingScheme', 'masterCall', 'topic', 'topic_title',
       'broad_topic', 'Área'],
      dtype='str')

In [11]:
org_project_count = (
    topics_to_devide_df.groupby("organisationID")["projectID"]
      .nunique()
      .rename("org_project_count")
)
topics_to_devide_df = topics_to_devide_df.merge(org_project_count, on="organisationID", how="left")

org_funding = topics_to_devide_df.groupby("organisationID").agg(
    org_total_ec_contribution=("ecContribution", "sum"),
    org_avg_ec_contribution=("ecContribution", "mean")
).reset_index()

topics_to_devide_df = topics_to_devide_df.merge(org_funding, on="organisationID", how="left")

topics_to_devide_df["_is_leader"] = (
    topics_to_devide_df["role"].astype(str).str.lower().str.contains("coord")
)

org_lead = (
    topics_to_devide_df.groupby("organisationID")["_is_leader"]
      .mean()
      .rename("org_lead_rate")
      .reset_index()
)

topics_to_devide_df = topics_to_devide_df.merge(org_lead, on="organisationID", how="left")
topics_to_devide_df.drop(columns="_is_leader", inplace=True)

area_counts = (
    topics_to_devide_df.groupby(["organisationID", "Área"])["projectID"]
      .nunique()
      .reset_index(name="n")
)

# --- Dominant area (label + count)
org_main_area_df = (
    area_counts.sort_values(["organisationID", "n"], ascending=[True, False])
              .drop_duplicates("organisationID")
              .rename(columns={"Área": "org_main_area", "n": "org_main_area_n"})
              [["organisationID", "org_main_area", "org_main_area_n"]]
)
topics_to_devide_df = topics_to_devide_df.merge(org_main_area_df, on="organisationID", how="left")

# --- Focus
tot = area_counts.groupby("organisationID")["n"].sum().reset_index(name="tot")
mx = area_counts.groupby("organisationID")["n"].max().reset_index(name="mx")

focus = tot.merge(mx, on="organisationID")
focus["org_area_focus"] = focus["mx"] / focus["tot"]

topics_to_devide_df = topics_to_devide_df.merge(
    focus[["organisationID","org_area_focus"]],
    on="organisationID",
    how="left"
)

# --- Entropy
entropy_df = (
    area_counts.groupby("organisationID")
    .apply(lambda x: entropy(x["n"]/x["n"].sum(), base=2) if len(x)>1 else 0)
    .reset_index(name="org_area_entropy")
)

topics_to_devide_df = topics_to_devide_df.merge(entropy_df, on="organisationID", how="left")

topic_concentration = (
    topics_to_devide_df.groupby("projectID")["Área"]
      .nunique()
      .rename("project_area_count")
      .reset_index()
)

topics_to_devide_df = topics_to_devide_df.merge(topic_concentration, on="projectID", how="left")

In [12]:
bad_orgs = (topics_to_devide_df.groupby("name")["country"]
            .apply(lambda s: s.dropna().astype(str).str.strip().replace("", np.nan).dropna().empty))

print("Organisations with no valid country:", bad_orgs.sum())
display(bad_orgs[bad_orgs].head(20))

country_fix_iso2 = {
    "CEREGE": "FR",
    "Kuzikus African Safaris PTY LTD": "NA",
    "MINISTRY OF HEALTH AND SOCIAL SERVICES": "NA",
    "NAMIBIA UNIVERSITY OF SCIENCE AND TECHNOLOGY": "NA",
    "ONGWE MINERALS (PTY) LTD": "NA",
    "Purdue University": "US",
    "Rijk Zwaan": "NL",
    "SADC CENTRE FOR RENEWABLE ENERGY AND ENERGY EFFICIENCY (SACREEE)": "NA",
    "UNIVERSITY OF NAMIBIA": "NA",
    "US National Library of Medicine": "US",
    "Université de Brest": "FR",
    "Virgin Atlantic Airways Ltd": "GB"
}
mask = topics_to_devide_df["country"].isna()
topics_to_devide_df.loc[mask, "country"] = topics_to_devide_df.loc[mask, "name"].map(country_fix_iso2)

Organisations with no valid country: 12


name
CEREGE                                                              True
Kuzikus African Safaris PTY LTD                                     True
MINISTRY OF HEALTH AND SOCIAL SERVICES                              True
NAMIBIA UNIVERSITY OF SCIENCE AND TECHNOLOGY                        True
ONGWE MINERALS (PTY) LTD                                            True
Purdue University                                                   True
Rijk Zwaan                                                          True
SADC CENTRE FOR RENEWABLE ENERGY AND ENERGY EFFICIENCY (SACREEE)    True
UNIVERSITY OF NAMIBIA                                               True
US National Library of Medicine                                     True
Université de Brest                                                 True
Virgin Atlantic Airways Ltd                                         True
Name: country, dtype: bool

In [13]:
topics_to_devide_df

,projectID,projectAcronym,organisationID,name,SME,activityType,country,nutsCode,geolocation,ecContribution,...,Área,org_project_count,org_total_ec_contribution,org_avg_ec_contribution,org_lead_rate,org_main_area,org_main_area_n,org_area_focus,org_area_entropy,project_area_count
0,101201611,COOLIO,999869114,UNIVERSITAET INNSBRUCK,False,HES,AT,AT332,"47.2629172,11.3843965",2500000.00,...,Ciências Fundamentais,62,3.341666e+07,6.305031e+05,0.306452,Digital & Indústria,15,0.241935,2.389654,1
1,101198761,MINTRAF,998922879,CENTRO NACIONAL DE INVESTIGACIONES CARDIOVASCU...,False,REC,ES,ES300,"40.47668785,-3.696091431284352",2455167.00,...,Ciências Fundamentais,11,9.045707e+06,8.223370e+05,0.636364,Saúde,6,0.545455,1.435371,1
2,101198411,StereoCPC,999907720,TECHNION - ISRAEL INSTITUTE OF TECHNOLOGY,False,HES,IL,IL,"32.8191218,34.9983856",2500000.00,...,Ciências Fundamentais,105,9.155998e+07,1.028764e+06,0.533333,Ciências Fundamentais,43,0.409524,2.032802,1
3,101199868,DM-Dawn,999596544,PHYSIKALISCH-TECHNISCHE BUNDESANSTALT,False,REC,DE,DE911,"52.2950801,10.4578976486826",399668.75,...,Ciências Fundamentais,17,6.221549e+06,4.147699e+05,0.117647,Digital & Indústria,8,0.470588,2.043058,1
4,101199868,DM-Dawn,999979306,WEIZMANN INSTITUTE OF SCIENCE,False,HES,IL,IL,"31.8952532,34.8105616",1529800.25,...,Ciências Fundamentais,128,1.660946e+08,1.350363e+06,0.718750,Ciências Fundamentais,80,0.625000,1.604066,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
116462,101132079,Kaleidos,915420526,WEDO PROJECT INTELLIGENCE MADE EASYSL,True,PRC,ES,ES511,"41.370449300000004,2.1500219563810696",298625.00,...,Educação & Formação,8,2.765178e+06,3.950255e+05,0.125000,Saúde,4,0.500000,1.405639,1
116463,101132079,Kaleidos,988004074,FUNDACIO PRIVADA PARC DE RECERCA UAB,False,OTH,ES,ES511,"41.4910324,2.1374969",0.00,...,Educação & Formação,6,0.000000e+00,0.000000e+00,0.000000,Educação & Formação,2,0.333333,1.918296,1
116464,101132079,Kaleidos,999993953,ALMA MATER STUDIORUM - UNIVERSITA DI BOLOGNA,False,HES,IT,ITH55,"44.4968718,11.3524529",144673.75,...,Educação & Formação,316,1.417147e+08,4.820227e+05,0.348101,Digital & Indústria,84,0.265823,2.332501,1
116465,101132079,Kaleidos,999986484,UNIVERSITAT AUTONOMA DE BARCELONA,False,HES,ES,ES511,"41.4910324,2.1374969",183762.50,...,Educação & Formação,170,8.640919e+07,6.128311e+05,0.423529,Educação & Formação,57,0.335294,2.260249,1


In [14]:
project_countries = (
    topics_to_devide_df.groupby("projectID")["country"]
    .apply(lambda x: set(x.dropna()))
    .reset_index(name="project_countries")
)
def safe_mode(s):
    s = s.dropna()
    return s.mode().iloc[0] if not s.mode().empty else np.nan

org_home_country = (
    topics_to_devide_df.groupby("organisationID")["country"]
    .agg(safe_mode)
    .rename("org_home_country")
    .reset_index()
)

In [15]:
df = topics_to_devide_df.merge(project_countries, on="projectID", how="left")
df = df.merge(org_home_country, on="organisationID", how="left")

In [16]:
def is_cross_border(row):
    if pd.isna(row["org_home_country"]):
        return 0
    return int(any(c != row["org_home_country"] for c in row["project_countries"]))

df["org_cross_border"] = df.apply(is_cross_border, axis=1)

In [17]:
org_cross_border_summary = (
    df.groupby("organisationID")["org_cross_border"]
    .agg(
        org_has_cross_border="max",
        org_cross_border_rate="mean"
    )
    .reset_index()
)

In [18]:
df = df.merge(org_cross_border_summary, on="organisationID", how="left")

In [19]:
df["year"] = pd.to_datetime(df["startDate"]).dt.year

org_year_funding = (
    df.groupby(["organisationID","year"])["ecContribution"]
    .sum()
    .reset_index()
)

org_funding_growth = (
    org_year_funding.groupby("organisationID")["ecContribution"]
    .apply(lambda s: s.pct_change().mean())
    .rename("org_funding_growth")
    .reset_index()
)
df = df.merge(org_funding_growth, on="organisationID", how="left")

In [20]:
df.columns 

Index(['projectID', 'projectAcronym', 'organisationID', 'name', 'SME',
       'activityType', 'country', 'nutsCode', 'geolocation', 'ecContribution',
       'totalCost_x', 'role', 'title', 'startDate', 'endDate', 'totalCost_y',
       'ecMaxContribution', 'objective', 'keywords', 'frameworkProgramme',
       'topics', 'fundingScheme', 'masterCall', 'topic', 'topic_title',
       'broad_topic', 'Área', 'org_project_count', 'org_total_ec_contribution',
       'org_avg_ec_contribution', 'org_lead_rate', 'org_main_area',
       'org_main_area_n', 'org_area_focus', 'org_area_entropy',
       'project_area_count', 'project_countries', 'org_home_country',
       'org_cross_border', 'org_has_cross_border', 'org_cross_border_rate',
       'year', 'org_funding_growth'],
      dtype='str')

In [21]:
df["start_year"] = pd.to_datetime(df["startDate"], errors="coerce").dt.year
df["end_year"] = pd.to_datetime(df["endDate"], errors="coerce").dt.year

In [22]:
org_duration = (
    df.groupby("organisationID")
      .agg(
          org_first_start_year=("start_year", "min"),
          org_last_end_year=("end_year", "max")
      )
      .reset_index()
)

org_duration["org_eu_funding_duration_years"] = (
    org_duration["org_last_end_year"] - org_duration["org_first_start_year"]
)

df = df.merge(
    org_duration[["organisationID", "org_eu_funding_duration_years"]],
    on="organisationID",
    how="left"
)

In [23]:
money_cols = [
    "ecContribution",
    "org_total_ec_contribution",
    "org_avg_ec_contribution",
    "totalCost_x",
    "totalCost_y",
    "ecMaxContribution",
]

for c in money_cols:
    if c in df.columns:
        # converte para string, limpa espaços e separadores comuns, depois força numérico
        df[c] = (
            df[c].astype(str)
                .str.replace("\u00a0", "", regex=False)  # non-breaking space
                .str.replace(" ", "", regex=False)
                .str.replace(",", ".", regex=False)      # se tiver vírgula decimal
        )
        # remove tudo que não seja dígito, ponto ou sinal (caso tenha "€" etc.)
        df[c] = df[c].str.replace(r"[^0-9\.\-]", "", regex=True)

        df[c] = pd.to_numeric(df[c], errors="coerce")

In [24]:
df["ec_contribution_million"] = df["ecContribution"] / 1_000_000
df["org_total_ec_contribution_million"] = df["org_total_ec_contribution"] / 1_000_000
df["org_avg_ec_contribution_million"] = df["org_avg_ec_contribution"] / 1_000_000

df["total_cost_org_million"] = df["totalCost_x"] / 1_000_000
df["total_cost_project_million"] = df["totalCost_y"] / 1_000_000
df["ec_max_contribution_million"] = df["ecMaxContribution"] / 1_000_000

df["org_percentage_projects_in_main_area"] = df["org_area_focus"] * 100
df["org_cross_border_rate_percentage"] = df["org_cross_border_rate"] * 100
df["org_funding_growth_percentage"] = df["org_funding_growth"] * 100

df = df.rename(columns={
    "Área": "area",
    "org_main_area_n": "org_projects_in_main_area",
    "project_area_count": "distinct_project_areas",
    "org_has_cross_border": "org_worked_cross_border"
})

cols_to_drop = [
    "nutsCode",
    "geolocation",
    "title",
    "frameworkProgramme",
    "fundingScheme",
    "topics",
    "topic_title",
    "broad_topic",
    "org_home_country",
    "ecContribution",
    "org_total_ec_contribution",
    "org_avg_ec_contribution",
    "totalCost_x",
    "totalCost_y",
    "ecMaxContribution",
    "org_area_focus",
    "org_cross_border_rate",
    "org_funding_growth",
    "startDate",
    "endDate"
]

df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

In [25]:
df.columns

Index(['projectID', 'projectAcronym', 'organisationID', 'name', 'SME',
       'activityType', 'country', 'role', 'objective', 'keywords',
       'masterCall', 'topic', 'area', 'org_project_count', 'org_lead_rate',
       'org_main_area', 'org_projects_in_main_area', 'org_area_entropy',
       'distinct_project_areas', 'project_countries', 'org_cross_border',
       'org_worked_cross_border', 'year', 'start_year', 'end_year',
       'org_eu_funding_duration_years', 'ec_contribution_million',
       'org_total_ec_contribution_million', 'org_avg_ec_contribution_million',
       'total_cost_org_million', 'total_cost_project_million',
       'ec_max_contribution_million', 'org_percentage_projects_in_main_area',
       'org_cross_border_rate_percentage', 'org_funding_growth_percentage'],
      dtype='str')

In [26]:
pd.set_option("display.max_columns", None)
df

,projectID,projectAcronym,organisationID,name,SME,activityType,country,role,objective,keywords,masterCall,topic,area,org_project_count,org_lead_rate,org_main_area,org_projects_in_main_area,org_area_entropy,distinct_project_areas,project_countries,org_cross_border,org_worked_cross_border,year,start_year,end_year,org_eu_funding_duration_years,ec_contribution_million,org_total_ec_contribution_million,org_avg_ec_contribution_million,total_cost_org_million,total_cost_project_million,ec_max_contribution_million,org_percentage_projects_in_main_area,org_cross_border_rate_percentage,org_funding_growth_percentage
0,101201611,COOLIO,999869114,UNIVERSITAET INNSBRUCK,False,HES,AT,coordinator,Quantum physics lies at the heart of many phen...,"quantum simulation, quantum dynamics, quantum...",ERC-2024-ADG,ERC-2024-ADG,Ciências Fundamentais,62,0.306452,Digital & Indústria,15,2.389654,1,{AT},0,1,2025,2025,2030,9,2.500000,33.416663,0.630503,0.000000,0.000000,2.500000,24.193548,74.193548,649.151371
1,101198761,MINTRAF,998922879,CENTRO NACIONAL DE INVESTIGACIONES CARDIOVASCU...,False,REC,ES,coordinator,Mitochondrial research has consistently yielde...,"Mitochondria, OxPhos, Cardiovascular, Heter...",ERC-2024-ADG,ERC-2024-ADG,Ciências Fundamentais,11,0.636364,Saúde,6,1.435371,1,{ES},0,1,2025,2025,2030,8,2.455167,9.045707,0.822337,0.000000,0.000000,2.455167,54.545455,45.454545,247.423395
2,101198411,StereoCPC,999907720,TECHNION - ISRAEL INSTITUTE OF TECHNOLOGY,False,HES,IL,coordinator,Stereoselective synthesis is a central aspect ...,"Non-classical carbocation, stereochemistry, ac...",ERC-2024-ADG,ERC-2024-ADG,Ciências Fundamentais,105,0.533333,Ciências Fundamentais,43,2.032802,1,{IL},0,1,2025,2025,2030,8,2.500000,91.559977,1.028764,0.000000,0.000000,2.500000,40.952381,47.619048,-20.942307
3,101199868,DM-Dawn,999596544,PHYSIKALISCH-TECHNISCHE BUNDESANSTALT,False,REC,DE,participant,What is the nature of dark matter (DM)? This ...,"Nuclear clock, ultralight DM",ERC-2024-ADG,ERC-2024-ADG,Ciências Fundamentais,17,0.117647,Digital & Indústria,8,2.043058,1,"{IL, DE}",1,1,2025,2025,2030,8,0.399669,6.221549,0.414770,0.000000,0.000000,2.271204,47.058824,94.117647,-8.615734
4,101199868,DM-Dawn,999979306,WEIZMANN INSTITUTE OF SCIENCE,False,HES,IL,coordinator,What is the nature of dark matter (DM)? This ...,"Nuclear clock, ultralight DM",ERC-2024-ADG,ERC-2024-ADG,Ciências Fundamentais,128,0.718750,Ciências Fundamentais,80,1.604066,1,"{IL, DE}",1,1,2025,2025,2030,9,1.529800,166.094590,1.350363,0.000000,0.000000,2.271204,62.500000,32.031250,-22.628716
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
116462,101132079,Kaleidos,915420526,WEDO PROJECT INTELLIGENCE MADE EASYSL,True,PRC,ES,coordinator,KALEIDOS brings together research and innovati...,"Knowledge Valorisation, Open Science, Quadrupl...",HORIZON-WIDERA-2023-ERA-01,HORIZON-WIDERA-2023-ERA-01-03,Educação & Formação,8,0.125000,Saúde,4,1.405639,1,"{IT, ES, IL, SE, FR}",1,1,2024,2024,2026,9,0.298625,2.765178,0.395025,0.298625,1.069225,1.069225,50.000000,100.000000,21.323287
116463,101132079,Kaleidos,988004074,FUNDACIO PRIVADA PARC DE RECERCA UAB,False,OTH,ES,thirdParty,KALEIDOS brings together research and innovati...,"Knowledge Valorisation, Open Science, Quadrupl...",HORIZON-WIDERA-2023-ERA-01,HORIZON-WIDERA-2023-ERA-01-03,Educação & Formação,6,0.000000,Educação & Formação,2,1.918296,1,"{IT, ES, IL, SE, FR}",1,1,2024,2024,2026,7,0.000000,0.000000,0.000000,0.000000,1.069225,1.069225,33.333333,100.000000,NaN
116464,101132079,Kaleidos,999993953,ALMA MATER STUDIORUM - UNIVERSITA DI BOLOGNA,False,HES,IT,participant,KALEIDOS brings together research and innovati...,"Knowledge Valorisation, Open Science, Quadrupl...",HORIZON-WIDERA-2023-ERA-01,HORIZON-WIDERA-2023-ERA-01-03,Educação & Formação,316,0.348101,Digital & Indústria,84,2.332501,1,"{IT, ES, IL, SE, FR}",1,1,2024,2024,2026,9,0.144674,141.714677,0.482023,0.144674,1

In [27]:
df["org_lead_rate_percentage"] = df["org_lead_rate"] * 100

percentage_cols = [
    "org_percentage_projects_in_main_area",
    "org_cross_border_rate_percentage",
    "org_funding_growth_percentage",
    "org_lead_rate_percentage"
]

for c in percentage_cols:
    if c in df.columns:
        df[c] = df[c].round(2)
        
df["ecContribution"] = df["ec_contribution_million"] * 1_000_000
df["org_total_ec_contribution"] = df["org_total_ec_contribution_million"] * 1_000_000
df["org_avg_ec_contribution"] = df["org_avg_ec_contribution_million"] * 1_000_000

df["totalCost_x"] = df["total_cost_org_million"] * 1_000_000
df["totalCost_y"] = df["total_cost_project_million"] * 1_000_000
df["ecMaxContribution"] = df["ec_max_contribution_million"] * 1_000_000

euro_cols = [
    "ecContribution",
    "org_total_ec_contribution",
    "org_avg_ec_contribution",
    "totalCost_x",
    "totalCost_y",
    "ecMaxContribution",
]

df[euro_cols] = df[euro_cols].round(0).astype("Int64")

cols_million = [
    "ec_contribution_million",
    "org_total_ec_contribution_million",
    "org_avg_ec_contribution_million",
    "total_cost_org_million",
    "total_cost_project_million",
    "ec_max_contribution_million",
]

df = df.drop(columns=[c for c in cols_million if c in df.columns])

In [28]:
df.columns 

Index(['projectID', 'projectAcronym', 'organisationID', 'name', 'SME',
       'activityType', 'country', 'role', 'objective', 'keywords',
       'masterCall', 'topic', 'area', 'org_project_count', 'org_lead_rate',
       'org_main_area', 'org_projects_in_main_area', 'org_area_entropy',
       'distinct_project_areas', 'project_countries', 'org_cross_border',
       'org_worked_cross_border', 'year', 'start_year', 'end_year',
       'org_eu_funding_duration_years', 'org_percentage_projects_in_main_area',
       'org_cross_border_rate_percentage', 'org_funding_growth_percentage',
       'org_lead_rate_percentage', 'ecContribution',
       'org_total_ec_contribution', 'org_avg_ec_contribution', 'totalCost_x',
       'totalCost_y', 'ecMaxContribution'],
      dtype='str')

In [29]:
final_column_order = [
    # Identifiers project
    "projectID",
    "projectAcronym",
    
    # Project structure & international context & governance
    "distinct_project_areas",
    "project_countries",
    "masterCall",
    "topic",
    "area",
    "objective",
    "keywords",

    # Organization characteristics / identifiers
    "organisationID",
    "name",
    "SME",
    "activityType",
    "role",
    "country",
    "org_cross_border",

    # Time & lifecycle
    "start_year",
    "end_year",
    "year",
    "org_eu_funding_duration_years",
        
    # Financial variables
    "ecContribution",
    "org_total_ec_contribution",
    "org_avg_ec_contribution",
    "totalCost_x",
    "totalCost_y",
    "ecMaxContribution",

    # Organization experience & positioning
    "org_project_count",
    "org_main_area",
    "org_projects_in_main_area",
    "org_percentage_projects_in_main_area",
    "org_area_entropy",

    # Leadership & internationalization strategy
    "org_lead_rate_percentage",
    "org_worked_cross_border",
    "org_cross_border_rate_percentage",

    
    # Dynamic trajectory
    "org_funding_growth_percentage",
]

# keep only columns that actually exist
final_column_order = [c for c in final_column_order if c in df.columns]

# reorder dataframe
df = df[final_column_order]

In [30]:
for i, col in enumerate(df.columns, 1):
    print(f"{i:02d} – {col}")

01 – projectID
02 – projectAcronym
03 – distinct_project_areas
04 – project_countries
05 – masterCall
06 – topic
07 – area
08 – objective
09 – keywords
10 – organisationID
11 – name
12 – SME
13 – activityType
14 – role
15 – country
16 – org_cross_border
17 – start_year
18 – end_year
19 – year
20 – org_eu_funding_duration_years
21 – ecContribution
22 – org_total_ec_contribution
23 – org_avg_ec_contribution
24 – totalCost_x
25 – totalCost_y
26 – ecMaxContribution
27 – org_project_count
28 – org_main_area
29 – org_projects_in_main_area
30 – org_percentage_projects_in_main_area
31 – org_area_entropy
32 – org_lead_rate_percentage
33 – org_worked_cross_border
34 – org_cross_border_rate_percentage
35 – org_funding_growth_percentage


In [31]:
#take the "#" out if you want to create a new dataset (already done and in use on the next notebook)

#df.to_csv("final_dataset_horizon_marketing.csv", index=False)

In [32]:
#map

In [33]:
#merged_df

In [34]:
!pip install plotly

In [35]:
import pandas as pd
import itertools
import plotly.graph_objects as go

In [36]:
df_merged = organization_df.merge(projects_df, left_on="projectID", right_on="id", how="left")

In [37]:
import pandas as pd
import itertools
import plotly.graph_objects as go

# -----------------------------
# 1. Merge organization + project info
# -----------------------------
df_merged = organization_df.merge(
    projects_df,
    left_on="projectID",
    right_on="id",
    how="left"
)

# -----------------------------
# 2. Split geolocation if needed
# -----------------------------
if "geolocation" in df_merged.columns and (
    "latitude" not in df_merged.columns or "longitude" not in df_merged.columns
):
    geo_split = df_merged["geolocation"].astype(str).str.split(",", expand=True)
    df_merged["latitude"] = pd.to_numeric(geo_split[0], errors="coerce")
    df_merged["longitude"] = pd.to_numeric(geo_split[1], errors="coerce")

# -----------------------------
# 3. Keep only valid geolocated rows
# -----------------------------
df_geo = df_merged.dropna(subset=["latitude", "longitude", "projectID", "organisationID"]).copy()

# Optional label
if "name" not in df_geo.columns:
    df_geo["name"] = df_geo["organisationID"].astype(str)

# One row per org-project pair
df_unique = df_geo.drop_duplicates(subset=["projectID", "organisationID"]).copy()

# -----------------------------
# 4. Sample 5% of organizations
# -----------------------------
unique_orgs = df_unique["organisationID"].drop_duplicates()
sampled_orgs = unique_orgs.sample(frac=0.05, random_state=42)

df_sample = df_unique[df_unique["organisationID"].isin(sampled_orgs)].copy()

# -----------------------------
# 5. Build edges:
# edge = partnership inside same project
# -----------------------------
edges = []

for project_id, group in df_sample.groupby("projectID"):
    orgs = group[["organisationID", "name", "latitude", "longitude"]].drop_duplicates()

    # only keep projects where at least 2 sampled orgs are present
    if len(orgs) >= 2:
        for org1, org2 in itertools.combinations(orgs.to_dict("records"), 2):
            # sort pair so A-B and B-A are treated as the same edge
            if org1["organisationID"] < org2["organisationID"]:
                source, target = org1, org2
            else:
                source, target = org2, org1

            edges.append({
                "projectID": project_id,
                "source_id": source["organisationID"],
                "source_name": source["name"],
                "source_lat": source["latitude"],
                "source_lon": source["longitude"],
                "target_id": target["organisationID"],
                "target_name": target["name"],
                "target_lat": target["latitude"],
                "target_lon": target["longitude"]
            })

edges_df = pd.DataFrame(edges)

print("Sampled organizations:", df_sample["organisationID"].nunique())
print("Raw edges:", len(edges_df))

# -----------------------------
# 6. Aggregate repeated collaborations
# -----------------------------
if not edges_df.empty:
    edges_agg = (
        edges_df.groupby(
            [
                "source_id", "target_id",
                "source_name", "target_name",
                "source_lat", "source_lon",
                "target_lat", "target_lon"
            ],
            as_index=False
        )
        .agg(shared_projects=("projectID", "nunique"))
    )
else:
    edges_agg = pd.DataFrame(columns=[
        "source_id", "target_id",
        "source_name", "target_name",
        "source_lat", "source_lon",
        "target_lat", "target_lon",
        "shared_projects"
    ])

print("Aggregated edges:", len(edges_agg))

# -----------------------------
# 7. Remove nodes with no partnerships
# keep only orgs that appear in at least one edge
# -----------------------------
if not edges_agg.empty:
    connected_ids = set(edges_agg["source_id"]).union(set(edges_agg["target_id"]))
    df_connected = df_sample[df_sample["organisationID"].isin(connected_ids)].copy()
else:
    df_connected = df_sample.iloc[0:0].copy()

# Build nodes only from connected organizations
nodes = (
    df_connected.groupby(
        ["organisationID", "name", "latitude", "longitude"],
        as_index=False
    )
    .agg(n_projects=("projectID", "nunique"))
)

print("Connected nodes:", len(nodes))

# -----------------------------
# 8. Build edge coordinates in ONE trace
# much lighter than one trace per edge
# -----------------------------
fig = go.Figure()

if not edges_agg.empty:
    edge_lons = []
    edge_lats = []
    edge_text = []

    for _, row in edges_agg.iterrows():
        edge_lons.extend([row["source_lon"], row["target_lon"], None])
        edge_lats.extend([row["source_lat"], row["target_lat"], None])
        edge_text.extend([
            f"{row['source_name']} ↔ {row['target_name']}<br>Shared projects: {row['shared_projects']}",
            f"{row['source_name']} ↔ {row['target_name']}<br>Shared projects: {row['shared_projects']}",
            None
        ])

    fig.add_trace(go.Scattergeo(
        lon=edge_lons,
        lat=edge_lats,
        mode="lines",
        line=dict(width=1, color="blue"),
        opacity=0.2,
        text=edge_text,
        hoverinfo="text"
    ))

# -----------------------------
# 9. Add node trace
# -----------------------------
if not nodes.empty:
    fig.add_trace(go.Scattergeo(
        lon=nodes["longitude"],
        lat=nodes["latitude"],
        mode="markers",
        marker=dict(
            size=6,
            color=nodes["n_projects"],
            colorscale="Viridis",
            showscale=True,
            colorbar=dict(title="Number of projects")
        ),
        text=nodes["name"] + "<br>Projects: " + nodes["n_projects"].astype(str),
        hoverinfo="text"
    ))

# -----------------------------
# 10. Layout
# -----------------------------
fig.update_layout(
    title="5% Sample of Organization Partnership Network",
    geo=dict(
        scope="world",
        showland=True,
        showcountries=True,
        showocean=True,
        projection_type="natural earth"
    ),
    height=700
)

# Better than fig.show() for large plots
fig.write_html("organization_partnership_map_5pct.html")
print("Saved to organization_partnership_map_5pct.html")

Sampled organizations: 1415
Raw edges: 1897
Aggregated edges: 1596
Connected nodes: 899
Saved to organization_partnership_map_5pct.html


In [38]:
import plotly.io as pio
pio.renderers.default = "notebook"